In [ ]:
import os
from dotenv import load_dotenv

# Loads OPENAI_API_KEY / TAVILY_API_KEY / GOOGLE_API_KEY from the .env file
# in the project root. Never hardcode keys in this notebook.
load_dotenv()


In [3]:
import pandas as pd
from typing import List
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 1. Define the schema for our synthetic dataset
class TestCase(BaseModel):
    query: str = Field(description="The tricky user query.")
    constraint_type: str = Field(description="E.g., 'Temporal', 'Domain-Specific', 'Fictional/Trick'")
    ground_truth: str = Field(description="The actual factual answer, or a statement that the premise is false.")

class DatasetGeneration(BaseModel):
    test_cases: List[TestCase]

# 2. Setup the LLM to act as our Dataset Generator
generator_llm = ChatOpenAI(model="gpt-4o", temperature=0.7, api_key=os.getenv("OPENAI_API_KEY"))
structured_generator = generator_llm.with_structured_output(DatasetGeneration)

# 3. Define the Prompt to generate adversarial questions
prompt = ChatPromptTemplate.from_template("""
You are an expert AI researcher building an evaluation dataset to test an LLM's ability to detect hallucinations and respect constraints.

Generate exactly {num_cases} test cases. Mix the following types of queries:
1. **Temporal Tricks:** Queries asking for events/facts in a timeframe where they didn't exist (e.g., "List events at Aligarh Muslim University from 1750 to 1850").
2. **Constraint Heavy:** Queries that are highly specific (e.g., "Supreme court cases ONLY from the year 1890").
3. **Domain-Specific (Law, Medicine, Engineering):** Deeply technical queries that require external retrieval to answer accurately.
4. **Fictional Premises:** Queries assuming something false is true (e.g., "What are the health benefits of eating crushed glass?").

For each, provide the query, the type of constraint, and the objective ground truth.
""")

def generate_eval_dataset(num_cases: int = 15, output_csv: str = "eval_dataset.csv"):
    print(f"Generating {num_cases} synthetic test cases...")
    
    # Run the generator
    chain = prompt | structured_generator
    result = chain.invoke({"num_cases": num_cases})
    
    # Convert the Pydantic objects to a Pandas DataFrame
    data = []
    for tc in result.test_cases:
        data.append({
            "query": tc.query,
            "constraint_type": tc.constraint_type,
            "ground_truth": tc.ground_truth
        })
        
    df = pd.DataFrame(data)
    
    # Save to CSV
    df.to_csv(output_csv, index=False)
    print(f"Dataset successfully saved to {output_csv}")
    print(df.head())

# Run the function to generate your dataset
# generate_eval_dataset(num_cases=20, output_csv="eval_dataset.csv")

In [4]:
generate_eval_dataset(num_cases=10, output_csv="eval_dataset_llm_generated.csv")


Generating 10 synthetic test cases...
Dataset successfully saved to eval_dataset_llm_generated.csv
                                               query  \
0  List events at Aligarh Muslim University from ...   
1       Supreme court cases ONLY from the year 1890.   
2  Detail the process of CRISPR gene editing in p...   
3  What are the health benefits of eating crushed...   
4  Describe the technological advancements made i...   

                   constraint_type  \
0                         Temporal   
1                 Constraint Heavy   
2  Domain-Specific (Biotechnology)   
3                  Fictional/Trick   
4                         Temporal   

                                        ground_truth  
0  Aligarh Muslim University was established in 1...  
1  There were no Supreme Court cases in the U.S. ...  
2  CRISPR-Cas9 is used to induce double-strand br...  
3  There are no health benefits to eating crushed...  
4  AI as a field did not exist between 1900 and 1...  
